# Chapter 27: Front End vs Back End

<a href="../lite/lab/index.html?path=ch27_frontend_backend.ipynb" target="_blank" style="display:inline-block;padding:8px 16px;background:#1976d2;color:white;border-radius:4px;text-decoration:none;font-weight:bold">▶ Open in JupyterLite (editable, no install)</a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import least_squares

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

Every SLAM system has two halves that barely speak to each other. The
**front end** looks at raw sensor data and extracts features, matches,
and associations. The **back end** takes those reports and solves a
giant optimization problem. When SLAM fails, the first question is
always: did the front end report garbage, or did the back end optimize
wrong?

This chapter builds a complete pipeline from scratch: a simulated 2D
world, a front end that detects landmarks (sometimes incorrectly), and
a back end that optimizes poses and map. We will watch the system work
perfectly, then break it, then fix it.

## 27.1 Perception vs Estimation: End to End Pipeline

We build a complete simulation:

1. **World:** 8 landmarks scattered in 2D.
2. **Robot:** Drives a loop with noisy odometry.
3. **Front end:** Detects landmarks within sensor range, adds noise,
   and occasionally assigns the **wrong landmark ID** (data association
   error).
4. **Back end:** Takes the front end output and runs least squares
   optimization to recover poses and landmark positions.

The measurement model is range/bearing from pose $i$ to landmark $j$:

$$\mathbf{z}_{ij} = \mathbf{m}_j - \mathbf{p}_i + \boldsymbol{\eta}, \quad \boldsymbol{\eta} \sim \mathcal{N}(\mathbf{0}, \sigma^2 \mathbf{I})$$

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
np.random.seed(42)
n_poses = 20                  # robot poses along the loop
n_landmarks = 8               # landmarks in the world
loop_radius = 6.0             # radius of the circular trajectory
sensor_range = 8.0            # max detection range
sigma_odom = 0.15             # odometry noise std
sigma_obs = 0.25              # observation noise std
# ─────────────────────────────────────────────────────────────────────────────

# Ground truth poses on a circle
angles = np.linspace(0, 2 * np.pi, n_poses, endpoint=False)
gt_poses = np.column_stack([loop_radius * np.cos(angles),
                            loop_radius * np.sin(angles)])

# Ground truth landmarks
gt_landmarks = np.array([
    [-4.0,  5.0], [ 3.0,  6.0], [ 7.0,  1.0], [ 5.0, -4.0],
    [-1.0, -6.0], [-6.0, -3.0], [-7.0,  2.0], [ 0.0,  0.0]
])

fig, ax = plt.subplots(figsize=(8, 8))
ax.plot(gt_poses[:, 0], gt_poses[:, 1], 'steelblue', lw=2, marker='o',
        ms=5, label='Robot path (ground truth)')
ax.scatter(gt_landmarks[:, 0], gt_landmarks[:, 1], c='tomato', s=120,
           marker='^', zorder=5, label='Landmarks')
for k in range(n_landmarks):
    ax.annotate(f'  L{k}', gt_landmarks[k], fontsize=10, color='tomato')
circle = plt.Circle((0, 0), sensor_range, fill=False, ls=':', color='gray',
                     label=f'Sensor range ({sensor_range} m) from origin')
ax.add_patch(circle)
ax.set_aspect('equal'); ax.legend(fontsize=10, loc='upper left')
ax.set_title('Simulated 2D SLAM world', fontsize=14)
plt.tight_layout(); plt.show()
print(f'{n_poses} poses, {n_landmarks} landmarks, sensor range = {sensor_range} m')

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
front_end_error_rate = 0.0    # 0% wrong associations for now
# ─────────────────────────────────────────────────────────────────────────────

def run_front_end(gt_poses, gt_landmarks, error_rate, sigma_obs,
                  sensor_range, rng=None):
    """Simulate front end: detect landmarks, add noise, maybe wrong IDs."""
    if rng is None:
        rng = np.random.default_rng(99)
    observations = []  # list of (pose_idx, landmark_idx, z_vec)
    n_lm = len(gt_landmarks)
    n_wrong = 0
    for i, p in enumerate(gt_poses):
        for j, m in enumerate(gt_landmarks):
            dist = np.linalg.norm(m - p)
            if dist < sensor_range:
                z = m - p + rng.normal(0, sigma_obs, 2)
                assoc = j
                if rng.random() < error_rate:
                    # Swap with a random other landmark
                    assoc = rng.integers(0, n_lm)
                    while assoc == j:
                        assoc = rng.integers(0, n_lm)
                    n_wrong += 1
                observations.append((i, assoc, z))
    return observations, n_wrong

# Build noisy odometry
odom_deltas = np.diff(gt_poses, axis=0)
odom_noise = np.random.randn(n_poses - 1, 2) * sigma_odom
noisy_odom = odom_deltas + odom_noise

# Dead reckoning from odometry
dr_poses = np.zeros_like(gt_poses)
dr_poses[0] = gt_poses[0]  # assume first pose known
for i in range(n_poses - 1):
    dr_poses[i + 1] = dr_poses[i] + noisy_odom[i]

obs_clean, n_wrong_clean = run_front_end(gt_poses, gt_landmarks, 0.0,
                                          sigma_obs, sensor_range)
print(f'Front end produced {len(obs_clean)} observations, '
      f'{n_wrong_clean} wrong associations')
print(f'Dead reckoning final error: '
      f'{np.linalg.norm(dr_poses[-1] - gt_poses[-1]):.3f} m')

In [ ]:
def back_end_optimize(init_poses, init_landmarks, observations,
                      odom_deltas, sigma_obs, sigma_odom, n_iter=30):
    """Least squares back end: optimize poses and landmarks jointly."""
    n_p = len(init_poses)
    n_m = len(init_landmarks)
    
    def pack(poses, landmarks):
        return np.concatenate([poses.ravel(), landmarks.ravel()])
    
    def unpack(x):
        poses = x[:2 * n_p].reshape(n_p, 2)
        landmarks = x[2 * n_p:].reshape(n_m, 2)
        return poses, landmarks
    
    def residuals(x):
        poses, lms = unpack(x)
        res = []
        # Odometry residuals
        for i in range(n_p - 1):
            pred = poses[i + 1] - poses[i]
            r = (pred - odom_deltas[i]) / sigma_odom
            res.extend(r)
        # Observation residuals
        for (pi, lj, z) in observations:
            pred = lms[lj] - poses[pi]
            r = (pred - z) / sigma_obs
            res.extend(r)
        # Anchor first pose
        res.extend((poses[0] - init_poses[0]) * 100.0)
        return np.array(res)
    
    x0 = pack(init_poses, init_landmarks)
    result = least_squares(residuals, x0, method='lm', max_nfev=n_iter * len(x0))
    opt_poses, opt_lms = unpack(result.x)
    return opt_poses, opt_lms, result.cost

# Initialize landmarks from observations (crude triangulation)
init_lms = np.zeros((n_landmarks, 2))
counts = np.zeros(n_landmarks)
for (pi, lj, z) in obs_clean:
    init_lms[lj] += dr_poses[pi] + z
    counts[lj] += 1
for j in range(n_landmarks):
    if counts[j] > 0:
        init_lms[j] /= counts[j]
    else:
        init_lms[j] = gt_landmarks[j] + np.random.randn(2)

opt_poses, opt_lms, cost = back_end_optimize(
    dr_poses, init_lms, obs_clean, noisy_odom, sigma_obs, sigma_odom)

print(f'Optimization cost: {cost:.4f}')
pose_err = np.mean(np.linalg.norm(opt_poses - gt_poses, axis=1))
lm_err = np.mean(np.linalg.norm(opt_lms - gt_landmarks, axis=1))
print(f'Mean pose error:     {pose_err:.4f} m')
print(f'Mean landmark error: {lm_err:.4f} m')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Dead reckoning
ax = axes[0]
ax.plot(gt_poses[:, 0], gt_poses[:, 1], 'forestgreen', ls='--', lw=1.5,
        alpha=0.5, label='Ground truth')
ax.plot(dr_poses[:, 0], dr_poses[:, 1], 'tomato', lw=2, marker='o',
        ms=4, label='Dead reckoning')
ax.scatter(gt_landmarks[:, 0], gt_landmarks[:, 1], c='gray', s=60,
           marker='^', alpha=0.5)
ax.set_aspect('equal'); ax.legend(fontsize=9)
ax.set_title('Step 0: Dead reckoning only', fontsize=12)

# After optimization (clean front end)
ax = axes[1]
ax.plot(gt_poses[:, 0], gt_poses[:, 1], 'forestgreen', ls='--', lw=1.5,
        alpha=0.5, label='True poses')
ax.plot(opt_poses[:, 0], opt_poses[:, 1], 'steelblue', lw=2, marker='o',
        ms=4, label='Optimized poses')
ax.scatter(gt_landmarks[:, 0], gt_landmarks[:, 1], c='forestgreen',
           s=60, marker='^', alpha=0.5, label='True landmarks')
ax.scatter(opt_lms[:, 0], opt_lms[:, 1], c='steelblue', s=80,
           marker='^', zorder=5, label='Optimized landmarks')
ax.set_aspect('equal'); ax.legend(fontsize=9)
ax.set_title('Step 1: Perfect front end + back end', fontsize=12)

# Show observation lines for one pose
ax = axes[2]
ax.plot(opt_poses[:, 0], opt_poses[:, 1], 'steelblue', lw=1.5, marker='o', ms=3)
ax.scatter(opt_lms[:, 0], opt_lms[:, 1], c='tomato', s=80, marker='^', zorder=5)
pose_idx = 5
for (pi, lj, z) in obs_clean:
    if pi == pose_idx:
        ax.plot([opt_poses[pi, 0], opt_lms[lj, 0]],
               [opt_poses[pi, 1], opt_lms[lj, 1]],
               'orange', lw=1.5, alpha=0.7)
ax.plot(opt_poses[pose_idx, 0], opt_poses[pose_idx, 1], 'o',
        color='orange', ms=10, zorder=6)
ax.set_aspect('equal')
ax.set_title(f'Observations from pose {pose_idx}', fontsize=12)

plt.suptitle('Full SLAM Pipeline: Front End + Back End', fontsize=14,
             fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

**Observation:** With a **perfect front end** (zero wrong associations),
the back end recovers both robot poses and landmark positions with high
accuracy. The optimization fuses odometry and landmark observations,
producing a result much better than dead reckoning alone. Now let us
see what happens when the front end makes mistakes.

## 27.2 Responsibilities: Same Back End, Different Front Ends

The key insight: the back end is **only as good as the data it receives**.
We now run the exact same optimizer with three different front end
quality levels:

| Front end | Error rate | What happens |
|-----------|-----------|---------------|
| **Perfect** | 0% | Accurate map |
| **Noisy** | 10% | Slight degradation |
| **Broken** | 30% | Map collapses |

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
error_rates = [0.0, 0.10, 0.30]
# ─────────────────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
results = {}

for idx, rate in enumerate(error_rates):
    rng = np.random.default_rng(42 + idx)
    obs, n_wrong = run_front_end(gt_poses, gt_landmarks, rate,
                                  sigma_obs, sensor_range, rng)
    
    # Initialize landmarks
    init_l = np.zeros((n_landmarks, 2))
    cnts = np.zeros(n_landmarks)
    for (pi, lj, z) in obs:
        init_l[lj] += dr_poses[pi] + z
        cnts[lj] += 1
    for j in range(n_landmarks):
        if cnts[j] > 0:
            init_l[j] /= cnts[j]
        else:
            init_l[j] = gt_landmarks[j] + rng.normal(0, 1, 2)
    
    op, ol, c = back_end_optimize(dr_poses, init_l, obs, noisy_odom,
                                   sigma_obs, sigma_odom)
    pe = np.mean(np.linalg.norm(op - gt_poses, axis=1))
    le = np.mean(np.linalg.norm(ol - gt_landmarks, axis=1))
    results[rate] = (pe, le, n_wrong)
    
    ax = axes[idx]
    ax.plot(gt_poses[:, 0], gt_poses[:, 1], 'forestgreen', ls='--',
            lw=1.5, alpha=0.4)
    colors = ['steelblue', 'orange', 'tomato']
    ax.plot(op[:, 0], op[:, 1], colors[idx], lw=2, marker='o', ms=4)
    ax.scatter(gt_landmarks[:, 0], gt_landmarks[:, 1], c='forestgreen',
              s=50, marker='^', alpha=0.4)
    ax.scatter(ol[:, 0], ol[:, 1], c=colors[idx], s=80, marker='^',
              zorder=5)
    ax.set_aspect('equal')
    label = ['Perfect', 'Noisy (10%)', 'Broken (30%)'][idx]
    ax.set_title(f'{label} front end\npose err = {pe:.3f} m, '
                 f'landmark err = {le:.3f} m', fontsize=11)

plt.suptitle('Same back end, different front end quality', fontsize=14,
             fontweight='bold', y=1.03)
plt.tight_layout(); plt.show()

In [ ]:
# Bar chart comparison
rates_pct = [f'{r:.0%}' for r in error_rates]
pose_errs = [results[r][0] for r in error_rates]
lm_errs = [results[r][1] for r in error_rates]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax = axes[0]
bars = ax.bar(rates_pct, pose_errs, color=['steelblue', 'orange', 'tomato'],
              alpha=0.8)
ax.set_xlabel('Front end error rate', fontsize=12)
ax.set_ylabel('Mean pose error (m)', fontsize=12)
ax.set_title('Pose accuracy degrades with front end errors', fontsize=12)
for b, v in zip(bars, pose_errs):
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.02,
            f'{v:.3f}', ha='center', fontsize=11)

ax = axes[1]
bars = ax.bar(rates_pct, lm_errs, color=['steelblue', 'orange', 'tomato'],
              alpha=0.8)
ax.set_xlabel('Front end error rate', fontsize=12)
ax.set_ylabel('Mean landmark error (m)', fontsize=12)
ax.set_title('Landmark accuracy degrades even faster', fontsize=12)
for b, v in zip(bars, lm_errs):
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.02,
            f'{v:.3f}', ha='center', fontsize=11)

plt.tight_layout(); plt.show()
print('The back end does its best, but garbage in means garbage out.')

**Key takeaway:** The back end optimizer is mathematically correct in all
three cases. It finds the best least squares solution given the
observations it received. The problem is that wrong associations create
conflicting constraints, and the optimizer compromises by distorting the
entire map. The lesson: **always suspect the front end first**.

## 27.3 Debugging: Residual Analysis

After optimization, each observation produces a **residual**: the
difference between the predicted and measured relative position. For
correct associations, residuals are small (near the noise level). For
wrong associations, residuals are large because the optimizer cannot
satisfy conflicting constraints.

$$r_{ij} = \| (\hat{\mathbf{m}}_j - \hat{\mathbf{p}}_i) - \mathbf{z}_{ij} \|$$

A histogram of residuals reveals the outliers.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
np.random.seed(77)
debug_error_rate = 0.15       # 15% wrong associations
residual_threshold = 1.5      # flag residuals above this (in sigma units)
# ─────────────────────────────────────────────────────────────────────────────

rng_dbg = np.random.default_rng(77)
obs_dbg, n_wrong_dbg = run_front_end(gt_poses, gt_landmarks,
                                      debug_error_rate, sigma_obs,
                                      sensor_range, rng_dbg)

# Track which observations are wrong (for validation)
rng_check = np.random.default_rng(77)
is_wrong = []
for i, p in enumerate(gt_poses):
    for j, m in enumerate(gt_landmarks):
        dist = np.linalg.norm(m - p)
        if dist < sensor_range:
            _ = rng_check.normal(0, sigma_obs, 2)
            flag = rng_check.random() < debug_error_rate
            if flag:
                _ = rng_check.integers(0, n_landmarks)
                while _ == j:
                    _ = rng_check.integers(0, n_landmarks)
            is_wrong.append(flag)

# Initialize and optimize
init_l_dbg = np.zeros((n_landmarks, 2))
cnts_dbg = np.zeros(n_landmarks)
for (pi, lj, z) in obs_dbg:
    init_l_dbg[lj] += dr_poses[pi] + z
    cnts_dbg[lj] += 1
for j in range(n_landmarks):
    if cnts_dbg[j] > 0:
        init_l_dbg[j] /= cnts_dbg[j]
    else:
        init_l_dbg[j] = gt_landmarks[j] + np.random.randn(2)

opt_p_dbg, opt_l_dbg, _ = back_end_optimize(
    dr_poses, init_l_dbg, obs_dbg, noisy_odom, sigma_obs, sigma_odom)

print(f'Observations: {len(obs_dbg)}, wrong: {n_wrong_dbg}')

In [ ]:
# Compute residuals for each observation
residuals_dbg = []
for (pi, lj, z) in obs_dbg:
    pred = opt_l_dbg[lj] - opt_p_dbg[pi]
    r = np.linalg.norm(pred - z) / sigma_obs
    residuals_dbg.append(r)
residuals_dbg = np.array(residuals_dbg)

# Color by correct/wrong
colors_res = ['steelblue' if not w else 'tomato' for w in is_wrong]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.bar(range(len(residuals_dbg)), residuals_dbg, color=colors_res, alpha=0.7)
ax.axhline(residual_threshold, color='orange', lw=2, ls='--',
           label=f'Threshold = {residual_threshold}')
ax.set_xlabel('Observation index', fontsize=11)
ax.set_ylabel('Normalized residual', fontsize=11)
ax.set_title('Residuals (blue = correct, red = wrong assoc.)', fontsize=12)
ax.legend(fontsize=10)

ax = axes[1]
correct_res = residuals_dbg[~np.array(is_wrong)]
wrong_res = residuals_dbg[np.array(is_wrong)]
ax.hist(correct_res, bins=20, color='steelblue', alpha=0.6,
        label='Correct associations')
if len(wrong_res) > 0:
    ax.hist(wrong_res, bins=15, color='tomato', alpha=0.6,
            label='Wrong associations')
ax.axvline(residual_threshold, color='orange', lw=2, ls='--',
           label=f'Threshold = {residual_threshold}')
ax.set_xlabel('Normalized residual', fontsize=11)
ax.set_ylabel('Count', fontsize=11)
ax.set_title('Residual histogram separates correct from wrong', fontsize=12)
ax.legend(fontsize=10)

plt.tight_layout(); plt.show()

flagged = residuals_dbg > residual_threshold
n_flagged = np.sum(flagged)
n_true_pos = np.sum(flagged & np.array(is_wrong))
n_false_pos = np.sum(flagged & ~np.array(is_wrong))
print(f'Flagged {n_flagged} observations as suspicious')
print(f'  True positives (actually wrong): {n_true_pos}')
print(f'  False positives (correct but flagged): {n_false_pos}')

In [ ]:
# Remove flagged observations and re-optimize
obs_cleaned = [obs_dbg[k] for k in range(len(obs_dbg))
               if not flagged[k]]

# Re-initialize landmarks
init_l_clean = np.zeros((n_landmarks, 2))
cnts_clean = np.zeros(n_landmarks)
for (pi, lj, z) in obs_cleaned:
    init_l_clean[lj] += dr_poses[pi] + z
    cnts_clean[lj] += 1
for j in range(n_landmarks):
    if cnts_clean[j] > 0:
        init_l_clean[j] /= cnts_clean[j]
    else:
        init_l_clean[j] = gt_landmarks[j] + np.random.randn(2)

opt_p_fix, opt_l_fix, _ = back_end_optimize(
    dr_poses, init_l_clean, obs_cleaned, noisy_odom, sigma_obs, sigma_odom)

# Compare before/after
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ax = axes[0]
ax.plot(gt_poses[:, 0], gt_poses[:, 1], 'forestgreen', ls='--', lw=1.5, alpha=0.4)
ax.plot(opt_p_dbg[:, 0], opt_p_dbg[:, 1], 'tomato', lw=2, marker='o', ms=4)
ax.scatter(opt_l_dbg[:, 0], opt_l_dbg[:, 1], c='tomato', s=80, marker='^', zorder=5)
ax.scatter(gt_landmarks[:, 0], gt_landmarks[:, 1], c='forestgreen',
           s=50, marker='^', alpha=0.4)
err_before = np.mean(np.linalg.norm(opt_p_dbg - gt_poses, axis=1))
ax.set_aspect('equal')
ax.set_title(f'Before fixing (error = {err_before:.3f} m)', fontsize=12)

ax = axes[1]
ax.plot(gt_poses[:, 0], gt_poses[:, 1], 'forestgreen', ls='--', lw=1.5, alpha=0.4)
ax.plot(opt_p_fix[:, 0], opt_p_fix[:, 1], 'steelblue', lw=2, marker='o', ms=4)
ax.scatter(opt_l_fix[:, 0], opt_l_fix[:, 1], c='steelblue', s=80, marker='^', zorder=5)
ax.scatter(gt_landmarks[:, 0], gt_landmarks[:, 1], c='forestgreen',
           s=50, marker='^', alpha=0.4)
err_after = np.mean(np.linalg.norm(opt_p_fix - gt_poses, axis=1))
ax.set_aspect('equal')
ax.set_title(f'After removing outliers (error = {err_after:.3f} m)', fontsize=12)

plt.suptitle('Residual analysis finds and fixes front end errors',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()
print(f'Error reduced from {err_before:.4f} to {err_after:.4f} m '
      f'({(1 - err_after/err_before)*100:.1f}% improvement)')

**Diagnostic strategy:**

1. Run the optimizer on all observations.
2. Compute the residual for every observation.
3. Flag observations with residuals above a threshold.
4. Remove flagged observations and re-optimize.
5. Repeat until no large residuals remain.

This iterative approach is called **robust estimation** or
**outlier rejection**. It relies on the fact that a few wrong
associations produce large residuals that stand out from the
distribution of correct observations.

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
n_rejection_rounds = 3
rejection_threshold = 1.2
# ─────────────────────────────────────────────────────────────────────────────

# Iterative outlier rejection
current_obs = list(obs_dbg)
history = []

for rnd in range(n_rejection_rounds):
    # Initialize landmarks
    il = np.zeros((n_landmarks, 2))
    ct = np.zeros(n_landmarks)
    for (pi, lj, z) in current_obs:
        il[lj] += dr_poses[pi] + z
        ct[lj] += 1
    for j in range(n_landmarks):
        if ct[j] > 0:
            il[j] /= ct[j]
        else:
            il[j] = gt_landmarks[j] + np.random.randn(2)
    
    op_r, ol_r, _ = back_end_optimize(
        dr_poses, il, current_obs, noisy_odom, sigma_obs, sigma_odom)
    
    # Compute residuals
    res_r = []
    for (pi, lj, z) in current_obs:
        pred = ol_r[lj] - op_r[pi]
        res_r.append(np.linalg.norm(pred - z) / sigma_obs)
    res_r = np.array(res_r)
    
    pe_r = np.mean(np.linalg.norm(op_r - gt_poses, axis=1))
    n_outliers = np.sum(res_r > rejection_threshold)
    history.append((rnd, len(current_obs), n_outliers, pe_r))
    
    # Remove outliers
    current_obs = [current_obs[k] for k in range(len(current_obs))
                   if res_r[k] <= rejection_threshold]
    
    print(f'Round {rnd}: {history[-1][1]} obs, {n_outliers} removed, '
          f'pose error = {pe_r:.4f} m')

# Final pass
il = np.zeros((n_landmarks, 2))
ct = np.zeros(n_landmarks)
for (pi, lj, z) in current_obs:
    il[lj] += dr_poses[pi] + z
    ct[lj] += 1
for j in range(n_landmarks):
    if ct[j] > 0:
        il[j] /= ct[j]
    else:
        il[j] = gt_landmarks[j] + np.random.randn(2)
op_final, ol_final, _ = back_end_optimize(
    dr_poses, il, current_obs, noisy_odom, sigma_obs, sigma_odom)
pe_final = np.mean(np.linalg.norm(op_final - gt_poses, axis=1))
print(f'Final: {len(current_obs)} obs, pose error = {pe_final:.4f} m')

In [ ]:
# Plot convergence of iterative rejection
rounds = [h[0] for h in history]
errors = [h[3] for h in history] + [pe_final]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(range(len(errors)), errors, 'steelblue', lw=2, marker='o', ms=8)
ax.set_xlabel('Rejection round', fontsize=12)
ax.set_ylabel('Mean pose error (m)', fontsize=12)
ax.set_title('Iterative outlier rejection convergence', fontsize=13)
ax.set_xticks(range(len(errors)))
plt.tight_layout(); plt.show()

---

## Capstone: Automatic SLAM Diagnostic Tool

Build a complete diagnostic function that takes SLAM output (estimated
poses, estimated landmarks, and raw observations) and automatically:

1. Computes all residuals
2. Flags suspicious observations
3. Reports which landmark associations are likely wrong
4. Re-optimizes with outliers removed
5. Compares before and after

We apply it to a **heavily corrupted** SLAM solution (25% wrong
associations) and show it can recover a usable map.

In [ ]:
def slam_diagnostic(opt_poses, opt_landmarks, observations, sigma_obs,
                    threshold=2.0):
    """
    Automatic SLAM diagnostic tool.
    
    Returns:
        report: dict with residuals, flags, and per-landmark statistics
    """
    n_obs = len(observations)
    residuals = np.zeros(n_obs)
    flagged = np.zeros(n_obs, dtype=bool)
    
    # Compute residuals
    for k, (pi, lj, z) in enumerate(observations):
        pred = opt_landmarks[lj] - opt_poses[pi]
        residuals[k] = np.linalg.norm(pred - z) / sigma_obs
        if residuals[k] > threshold:
            flagged[k] = True
    
    # Per-landmark statistics
    n_lm = len(opt_landmarks)
    lm_stats = {}
    for j in range(n_lm):
        mask = [obs[1] == j for obs in observations]
        lm_res = residuals[mask]
        if len(lm_res) > 0:
            lm_stats[j] = {
                'n_obs': len(lm_res),
                'mean_residual': np.mean(lm_res),
                'max_residual': np.max(lm_res),
                'n_flagged': np.sum(np.array(mask) & flagged)
            }
    
    return {
        'residuals': residuals,
        'flagged': flagged,
        'n_flagged': np.sum(flagged),
        'n_total': n_obs,
        'landmark_stats': lm_stats
    }

print('Diagnostic tool defined.')

In [ ]:
# ── PARAMETERS ── change these and re-run ────────────────────────────────────
np.random.seed(123)
capstone_error_rate = 0.25    # 25% wrong associations (heavy corruption)
diag_threshold = 1.5
n_fix_rounds = 4
# ─────────────────────────────────────────────────────────────────────────────

rng_cap = np.random.default_rng(123)
obs_cap, n_wrong_cap = run_front_end(gt_poses, gt_landmarks,
                                      capstone_error_rate, sigma_obs,
                                      sensor_range, rng_cap)

# Initial optimization (corrupted)
init_l_cap = np.zeros((n_landmarks, 2))
ct_cap = np.zeros(n_landmarks)
for (pi, lj, z) in obs_cap:
    init_l_cap[lj] += dr_poses[pi] + z
    ct_cap[lj] += 1
for j in range(n_landmarks):
    if ct_cap[j] > 0:
        init_l_cap[j] /= ct_cap[j]
    else:
        init_l_cap[j] = gt_landmarks[j] + np.random.randn(2)

opt_p_cap, opt_l_cap, _ = back_end_optimize(
    dr_poses, init_l_cap, obs_cap, noisy_odom, sigma_obs, sigma_odom)

print(f'Total observations: {len(obs_cap)}')
print(f'Wrong associations: {n_wrong_cap} ({n_wrong_cap/len(obs_cap)*100:.1f}%)')
err_cap_before = np.mean(np.linalg.norm(opt_p_cap - gt_poses, axis=1))
print(f'Initial pose error: {err_cap_before:.4f} m')

In [ ]:
# Run diagnostic and iterative fixing
current_obs_cap = list(obs_cap)
cap_history = []

for rnd in range(n_fix_rounds):
    # Optimize
    il_c = np.zeros((n_landmarks, 2))
    cc = np.zeros(n_landmarks)
    for (pi, lj, z) in current_obs_cap:
        il_c[lj] += dr_poses[pi] + z
        cc[lj] += 1
    for j in range(n_landmarks):
        if cc[j] > 0:
            il_c[j] /= cc[j]
        else:
            il_c[j] = gt_landmarks[j] + np.random.randn(2)
    
    op_c, ol_c, _ = back_end_optimize(
        dr_poses, il_c, current_obs_cap, noisy_odom, sigma_obs, sigma_odom)
    
    # Run diagnostic
    report = slam_diagnostic(op_c, ol_c, current_obs_cap, sigma_obs,
                             diag_threshold)
    
    pe_c = np.mean(np.linalg.norm(op_c - gt_poses, axis=1))
    cap_history.append((rnd, len(current_obs_cap), report['n_flagged'], pe_c))
    
    print(f'Round {rnd}: {len(current_obs_cap)} obs, '
          f'{report["n_flagged"]} flagged, pose error = {pe_c:.4f} m')
    
    # Print landmark report
    for lj, stats in sorted(report['landmark_stats'].items()):
        if stats['n_flagged'] > 0:
            print(f'    L{lj}: {stats["n_flagged"]}/{stats["n_obs"]} flagged, '
                  f'max residual = {stats["max_residual"]:.2f}')
    
    # Remove flagged
    current_obs_cap = [current_obs_cap[k] for k in range(len(current_obs_cap))
                       if not report['flagged'][k]]

# Final optimization
il_f = np.zeros((n_landmarks, 2))
cf = np.zeros(n_landmarks)
for (pi, lj, z) in current_obs_cap:
    il_f[lj] += dr_poses[pi] + z
    cf[lj] += 1
for j in range(n_landmarks):
    if cf[j] > 0:
        il_f[j] /= cf[j]
    else:
        il_f[j] = gt_landmarks[j] + np.random.randn(2)
op_cap_fix, ol_cap_fix, _ = back_end_optimize(
    dr_poses, il_f, current_obs_cap, noisy_odom, sigma_obs, sigma_odom)
err_cap_after = np.mean(np.linalg.norm(op_cap_fix - gt_poses, axis=1))
print(f'\nFinal: {len(current_obs_cap)} obs, pose error = {err_cap_after:.4f} m')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Before
ax = axes[0]
ax.plot(gt_poses[:, 0], gt_poses[:, 1], 'forestgreen', ls='--', lw=1.5, alpha=0.4)
ax.plot(opt_p_cap[:, 0], opt_p_cap[:, 1], 'tomato', lw=2, marker='o', ms=4)
ax.scatter(opt_l_cap[:, 0], opt_l_cap[:, 1], c='tomato', s=80, marker='^', zorder=5)
ax.scatter(gt_landmarks[:, 0], gt_landmarks[:, 1], c='forestgreen', s=50, marker='^', alpha=0.4)
ax.set_aspect('equal')
ax.set_title(f'Before diagnosis\n(25% wrong, error = {err_cap_before:.3f} m)', fontsize=11)

# After
ax = axes[1]
ax.plot(gt_poses[:, 0], gt_poses[:, 1], 'forestgreen', ls='--', lw=1.5, alpha=0.4)
ax.plot(op_cap_fix[:, 0], op_cap_fix[:, 1], 'steelblue', lw=2, marker='o', ms=4)
ax.scatter(ol_cap_fix[:, 0], ol_cap_fix[:, 1], c='steelblue', s=80, marker='^', zorder=5)
ax.scatter(gt_landmarks[:, 0], gt_landmarks[:, 1], c='forestgreen', s=50, marker='^', alpha=0.4)
ax.set_aspect('equal')
ax.set_title(f'After diagnosis\n(outliers removed, error = {err_cap_after:.3f} m)', fontsize=11)

# Convergence
ax = axes[2]
errs_cap = [h[3] for h in cap_history] + [err_cap_after]
ax.plot(range(len(errs_cap)), errs_cap, 'steelblue', lw=2, marker='o', ms=8)
ax.set_xlabel('Rejection round', fontsize=12)
ax.set_ylabel('Mean pose error (m)', fontsize=12)
ax.set_title('Convergence: error drops each round', fontsize=12)
ax.set_xticks(range(len(errs_cap)))

plt.suptitle('Capstone: Automatic SLAM Diagnostic Tool',
             fontsize=14, fontweight='bold', y=1.03)
plt.tight_layout(); plt.show()

improvement = (1 - err_cap_after / err_cap_before) * 100
print(f'\nDiagnostic tool reduced error by {improvement:.1f}%')
print(f'From {err_cap_before:.4f} m to {err_cap_after:.4f} m')

**Capstone observations:**

- The diagnostic tool successfully identifies most wrong associations
  using residual analysis alone, with no access to ground truth.
- Each round of outlier rejection improves the solution, which in turn
  makes the remaining outliers easier to detect.
- Even with 25% wrong associations, iterative diagnosis recovers a
  usable map.
- This is the core idea behind **robust SLAM**: separate the question
  "is this observation correct?" from the question "what is the best
  map given correct observations?"

---

## Exercises

### Exercise 27.1: Error Rate Sweep

Sweep the front end error rate from 0% to 50% in steps of 5%. For each
rate, run the full pipeline (without diagnostic fixing) and record the
mean pose error. Plot error vs. error rate. At what rate does the map
become unusable (error > 1 m)?

In [ ]:
# Your code here

### Exercise 27.2: Threshold Selection

The residual threshold is critical. Too low and you reject good
observations. Too high and you keep bad ones. Sweep the threshold
from 0.5 to 5.0 and plot (a) number of correctly removed outliers,
(b) number of incorrectly removed good observations, and (c) final
pose error. What is the optimal threshold for 15% error rate?

In [ ]:
# Your code here

### Exercise 27.3: Sensor Range vs. Robustness

A shorter sensor range means fewer observations per landmark, making
the system more sensitive to outliers. Sweep sensor range from 4 to 12
meters while keeping the error rate at 15%. Plot pose error vs. sensor
range, with and without the diagnostic tool.

In [ ]:
# Your code here

### Exercise 27.4: Chi Squared Test (challenge)

Instead of a fixed threshold, use the **chi squared test** to flag
outliers. For a 2D observation with known $\sigma$, the squared
normalized residual $r^2 / \sigma^2$ should follow a $\chi^2(2)$
distribution. Use the 95th percentile as the threshold. Compare this
approach to the fixed threshold in terms of true/false positive rates.

In [ ]:
# Your code here
# Hint: from scipy.stats import chi2
#        threshold_sq = chi2.ppf(0.95, df=2)